# Combining statistical plots

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://github.com/mggg/gerrytools/tree/main/user_guide/_static/data">Browse tutorial data</a></div>

The format guides show one builder at a time. This workflow turns the shared Georgia
tutorial data into two different analytical views, then places the finished builders on one
Matplotlib figure. The examples keep the plot-specific steps separate from the final layout.


In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gerrytools.plotting import BoxPlot, SeatsVotesPlot

data_dir = Path("data")
precincts = gpd.read_file(data_dir / "ga_2016_precincts.gpkg")
vote_columns = ["PRES16D", "PRES16R", "SEN16D", "SEN16R"]
precincts[vote_columns] = precincts[vote_columns].apply(pd.to_numeric)
district_votes = precincts.groupby("CD")[vote_columns].sum().sort_index()

The precinct file contains Democratic and Republican vote totals for the 2016 presidential
and U.S. Senate elections. Summing those columns by congressional district produces the
district-level inputs required by `SeatsVotesPlot`.

## Several elections on one seats-votes plot

`add_election()` expects target-party votes and corresponding totals for the same ordered
districts. Repeating the call keeps both elections on one set of axes, making their
uniform-swing curves directly comparable. The proportionality line is a shared reference,
so it is added once after the election loop.


In [ ]:
seats_votes = SeatsVotesPlot()
for name, party_column, opposition_column in [
    ("President", "PRES16D", "PRES16R"),
    ("U.S. Senate", "SEN16D", "SEN16R"),
]:
    party_votes = district_votes[party_column]
    total_votes = party_votes + district_votes[opposition_column]
    seats_votes.add_election(party_votes, total_votes, name)
seats_votes.add_proportionality_line()
seats_votes.show()

The two curves describe elections under one districting plan. The next view reverses that
perspective: it summarizes one demographic measure over many plans.

## A ranked ensemble distribution

Each ensemble record stores one BVAP share for every district in a plan. Sorting within each
record removes the arbitrary district labels. Column 1 then contains every plan's lowest-BVAP
district, column 2 the second-lowest, and so on. `BoxPlot` receives those rank distributions
as a mapping from displayed rank to observations.


In [ ]:
records = json.loads((data_dir / "ga_congressional_ensemble_a.json").read_text())
ranked = np.sort(np.asarray([record["BVAP"] for record in records]), axis=1)
ranked_distributions = {str(rank + 1): ranked[:, rank] for rank in range(ranked.shape[1])}

boxplot = BoxPlot()
boxplot.add_dataset(ranked_distributions)
boxplot.show()

The box at a given position therefore describes a rank, not a geographic district that can
be followed from plan to plan. This distinction matters whenever ensemble plans use unrelated
district labels.

## Put builders on one Matplotlib figure

Both builders are already configured and can be moved onto axes created by Matplotlib.
`bind_to_ax()` preserves their data and settings, transfers figure ownership to the caller,
and immediately renders each builder in its assigned panel. The single `plt.show()` displays
the completed figure once.


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 5))
seats_votes.bind_to_ax(axes[0])
boxplot.bind_to_ax(axes[1])
plt.show()

## Related

- [Statistical plots](index.md)
- [Shared statistical controls](options.ipynb)
- [Matplotlib composition](../composition.ipynb)
